In [ ]:
import os

data_dir = os.path.abspath("data/public")
project_dir = os.path.abspath("data/project")
chrombpnet_dir = os.path.join(project_dir, "chrombpnet")

In [ ]:
from chrombpnet.data import DefaultDataFile, get_default_data_path
import chrombpnet.helpers.preprocessing.auto_shift_detect as auto_shift_detect
import chrombpnet.training.utils.one_hot as one_hot
import pysam
import random
import pandas as pd
from collections import Counter
import subprocess
import os

### Preprocessing

In [ ]:
## Sample reads from tabix file
# Parameters
celltypes = ["acinar", "activated_stellate", "alpha", "beta", "cycling", "delta","ductal", "endothelial", "epsilon", "lymphoid", "mast", "myeloid", "pp", "schwann", "quiescent_stellate"]
chrom_size_file = f"{chrombpnet_dir}/data/hg38.chrom.sizes"
genome_fasta_path = f"{chrombpnet_dir}/data/hg38.fa"
n_sample = 10000

# Define the path for the fragment file
for celltype in celltypes:
    fragment_file = f"{data_dir}/scATACseq/ArchR/IsletProject/GroupFragments/fragments/" + str(celltype) + ".fragments.tsv.gz"
    output_bedgraph = f"{chrombpnet_dir}/data/" + str(celltype) + "_coverage.bedGraph"
    
    # Read the chromosome sizes
    chr_size = pd.read_csv(chrom_size_file, sep="\t", header=None)
    allowed_chroms = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
    chr_size = chr_size[chr_size[0].isin(allowed_chroms)].reset_index(drop=True)
    
    # Open the fragment file
    tb = pysam.TabixFile(fragment_file)

    # Sample using randomized regions
    sampled_reads = []
    counter = 0
    while counter <= n_sample:
        random_row = chr_size.sample(n=1)
        rand_num = random.randint(0, random_row.iloc[0,1]-1000)
        records = list(tb.fetch(random_row.iloc[0,0], rand_num, rand_num+1000))
        if records:
            read = random.choice(records).strip().split('\t')
            if counter == 0:
                sampled_reads = pd.DataFrame([read], index = [counter])
            else:
                sampled_reads = pd.concat((sampled_reads, pd.DataFrame([read], index = [counter])))
            counter += 1
    
    # Postprocessing
    sampled_reads[3] = 1000
    sampled_reads[4] = 0
    sampled_reads[5] = '+'
    sampled_reads = sampled_reads.rename({0:'chr', 1:'start', 2:'end', 3:'x1', 4:'x2', 5:'strand'}, axis=1)
    
    # Finalize
    plus_reads = sampled_reads.copy()
    plus_reads = plus_reads[plus_reads['strand']=="+"].iloc[:,:3]
    minus_reads = sampled_reads.copy()
    minus_reads['strand'] = "-"
    minus_reads = minus_reads[minus_reads['strand']=="-"].iloc[:,:3]
    
    # Calculate shifts
    genome_fasta_path = f"{chrombpnet_dir}/data/hg38.fa"
    plus_pwm, minus_pwm = auto_shift_detect.get_pwms(plus_reads, minus_reads, genome_fasta_path)
    ref_motifs_file=get_default_data_path(DefaultDataFile.atac_ref_motifs)
    ref_plus_pwms, ref_minus_pwms = auto_shift_detect.get_ref_pwms(ref_motifs_file)
    plus_shift, minus_shift = auto_shift_detect.compute_shift_ATAC(ref_plus_pwms, ref_minus_pwms, plus_pwm, minus_pwm)
    plus_shift_delta, minus_shift_delta = 4-plus_shift, -4-minus_shift
    
    # Summarize coverage
    coverage = Counter()
    for chrom in chr_size[0]:
        records = tb.fetch(chrom)
        for record in records:
            fields = record.strip().split('\t')
            chrom = fields[0]
            start = int(fields[1])
            end = int(fields[2])
            coverage[(chrom, max(0, start + plus_shift_delta))] += 1
            coverage[(chrom, max(0, end + minus_shift_delta - 1))] += 1
            last_pos_chrom = chrom
            last_pos_start = max(0, start + plus_shift_delta)
    
    # Close the fragment file
    tb.close()
    
    # Setup chromosomes
    chrom_sizes = {}
    for idx in chr_size.index:
        chrom_sizes[chr_size.iloc[idx][0]] = int(chr_size.iloc[idx][1])
    
    # Process
    with open(output_bedgraph, 'w') as out:
        for chrom in chrom_sizes:
            # Extract coverage for this chrom, sorted by position
            positions = sorted(pos for (c, pos) in coverage if c == chrom)
            if not positions:
                continue
            
            start = positions[0]
            prev_cov = coverage[(chrom, start)]
            end = start + 1  # bedGraph intervals are zero-based half-open: [start, end)
            
            for pos in positions[1:]:
                cov = coverage[(chrom, pos)]
                # Extend the current interval if coverage is same and positions are continuous
                if cov == prev_cov and pos == end:
                    end += 1
                else:
                    # Write the previous interval if coverage > 0
                    if prev_cov > 0:
                        out.write(f"{chrom}\t{start}\t{end}\t{prev_cov}\n")
                    start = pos
                    end = pos + 1
                    prev_cov = cov
            
            # Write last interval
            if prev_cov > 0:
                out.write(f"{chrom}\t{start}\t{end}\t{prev_cov}\n")

In [ ]:
# Sort and make bigWig
celltypes = ["acinar", "activated_stellate", "alpha", "beta", "cycling", "delta","ductal", "endothelial", "epsilon", "lymphoid", "mast", "myeloid", "pp", "schwann", "quiescent_stellate"]
chrom_size_file = f"{chrombpnet_dir}/data/hg38.chrom.sizes"
for celltype in celltypes:
    output_bedgraph = f"{chrombpnet_dir}/data/" + str(celltype) + "_coverage.bedGraph"
    sorted_bedgraph = f"{chrombpnet_dir}/data/" + str(celltype) + "_coverage_sorted.bedGraph"
    output_bigWig = f"{chrombpnet_dir}/data/" + str(celltype) + "_coverage_unstranded.bw"
    
    # Sort the bedGraph
    command = ['sort', '-k1,1', '-k2,2n', output_bedgraph]
    with open(sorted_bedgraph, 'w') as outfile:
        subprocess.run(command, stdout=outfile, check=True)
    
    # Make a bigWig
    subprocess.run(["bedGraphToBigWig", sorted_bedgraph, chrom_size_file, output_bigWig])

### Quality control the bigWigs

In [ ]:
import chrombpnet.helpers.preprocessing.analysis.build_pwm_from_bigwig as build_pwm_from_bigwig
import json
import argparse

In [ ]:
celltypes = ["acinar", "activated_stellate", "alpha", "beta", "cycling", "delta","ductal", "endothelial", "epsilon", "lymphoid", "mast", "myeloid", "pp", "schwann", "quiescent_stellate"]
genome_fasta_path = f"{chrombpnet_dir}/data/hg38.fa"
chrom_size_file = f"{chrombpnet_dir}/data/hg38.chrom.sizes"
for celltype in celltypes:
    for fold in range(5):
        # Make output directory
        output_directory = f"{chrombpnet_dir}/data/fold" + str(fold) + "/qc/"
        if not os.path.exists(output_directory):
            os.makedirs(output_directory)
            
        # Load the fold
        fold_path = f"{chrombpnet_dir}/data/fold_" + str(fold) + ".json"
        folds = json.load(open(fold_path))

        # Set the arguments
        args = argparse.Namespace(
            bigwig=f"{chrombpnet_dir}/data/" + str(celltype) + "_coverage_unstranded.bw",
            genome=genome_fasta_path,
            pwm_width=24,
            chrom_sizes = chrom_size_file,
            output_prefix = output_directory + str(celltype) + "_bigWig_qc",
            chr = folds["valid"][0]
        )

        # Call function
        build_pwm_from_bigwig.main(args)

### Merge all data for bias model training

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor
import glob
import os

In [ ]:
# List sorted bedGraph files
directory = f'{chrombpnet_dir}/data/'
pattern = '*sorted.bedGraph'
search_pattern = os.path.join(directory, pattern)
matching_files = glob.glob(search_pattern)

# Setup in a dictionary
keys = [path.split('/')[-1].split('_')[0] for path in matching_files]

# Create dictionary
files = dict(zip(keys, matching_files))

# Define command to run in parallel
def run_command(key, file):
    command = f"awk '{{print > \"{key}_\"$1\".bedGraph\"}}' {file}"
    try:
        subprocess.run(command, shell=True, check=True)
        print(f"Command for {key} executed successfully!")
    except subprocess.CalledProcessError as e:
        print(f"Error executing command for {key}: {e}")

# Run in parallel
with ThreadPoolExecutor() as executor:
    # Use executor.map to run the commands in parallel
    executor.map(lambda item: run_command(*item), files.items())

In [ ]:
# Define chromosomes
chromosomes = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]

# Define command to run in parallel
def run_command(chromosome):
    bedgraph = f"{chrombpnet_dir}/data/combined_" + str(chromosome) + ".bedGraph"
    command = ['bedtools', 'unionbedg', '-i']
    files = [f"{key}_{chromosome}.bedGraph" for key in keys]
    command.extend(files)
    with open(bedgraph, 'w') as outfile:
        result = subprocess.run(command, stdout=outfile, stderr=subprocess.PIPE, check=True)

# Run in parallel
with ThreadPoolExecutor() as executor:
    # Use executor.map to run the commands in parallel
    executor.map(run_command, chromosomes)

In [ ]:
# Define command to run in parallel
def run_command(chromosome):
    input_bedgraph = f"{chrombpnet_dir}/data/combined_" + str(chromosome) + ".bedGraph"
    output_bedgraph = f"{chrombpnet_dir}/data/combined_" + str(chromosome) + "_sum.bedGraph"
    
    # Open the input and output files
    with open(input_bedgraph, 'r') as infile, open(output_bedgraph, 'w') as outfile:
        for line in infile:
            # Split each line by whitespace (tabs or spaces)
            columns = line.split()
            
            # Extract the values from columns 4 onwards and sum them
            values_to_sum = map(int, columns[3:])  # Convert to integers
            total_sum = sum(values_to_sum)
            
            # Write the result to the output file
            result = f"{columns[0]}\t{columns[1]}\t{columns[2]}\t{total_sum}\n"
            outfile.write(result)

# Run in parallel
with ThreadPoolExecutor() as executor:
    # Use executor.map to run the commands in parallel
    executor.map(run_command, chromosomes)

In [ ]:
# Merge files
combined_sorted_file = f"{chrombpnet_dir}/data/combined_sum.bedGraph"
input_files = [f"{chrombpnet_dir}/data/combined_{chromosome}_sum.bedGraph" for chromosome in chromosomes]
command = ['cat'] + input_files
with open(combined_sorted_file, 'w') as outfile:
    try:
        result = subprocess.run(command, stdout=outfile, stderr=subprocess.PIPE, check=True)
        print(f"Concatenation for completed successfully!")
    except subprocess.CalledProcessError as e:
        print(f"Error concatenating files")

# Sort
combined_sorted_sum_file = f"{chrombpnet_dir}/data/combined_sum_sorted.bedGraph"
try:
    sort_command = f"sort -k1,1 -k2,2n {combined_sorted_file} > {combined_sorted_sum_file}"
    subprocess.run(sort_command, shell=True, check=True)
    print(f"Sorted file created successfully!")
except subprocess.CalledProcessError as e:
    print(f"Error sorting file")

# Convert to bigWig
output_bigWig = f"{chrombpnet_dir}/data/combined_unstranded.bw"
chrom_size_file = f"{chrombpnet_dir}/data/hg38.chrom.sizes"
try:
    subprocess.run(["bedGraphToBigWig", combined_sorted_sum_file, chrom_size_file, output_bigWig], stderr=subprocess.PIPE, check=True)
    print(f"bigWig conversion completed!")
except subprocess.CalledProcessError as e:
    print(f"Error making bigWig file")

### Functions for processing peaks

In [ ]:
## Source https://github.com/kundajelab/chrombpnet/blob/master/chrombpnet/helpers/make_gc_matched_negatives/get_gc_matched_negatives.py
import argparse
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import random
import csv 
import json
import sys

def remap_chrom(chrom, splits_dict):
    '''
    Remapping chromosome names - we will not differentiate between the train/valid/tes chromsomes
    when sampling negatives.
    '''
    if chrom in splits_dict["train"]:
        chrom_mod = "chrom_train"
    elif chrom in splits_dict["valid"]:
        chrom_mod = "chrom_valid"
    elif chrom in splits_dict["test"]:
        chrom_mod = "chrom_test"
    else:
        chrom_mod = "ignore"
    return chrom_mod

def make_gc_dict(candidate_negatives, splits_dict):
    """
    Imports the candidate negatives into a dictionary structure.
    The `key` is the gc content fraction, and the `values` are a list 
    containing the (chrom,start,end) of a region with the corresponding 
    gc content fraction.
    """
    data=open(candidate_negatives,'r').readlines()
    gc_dict={}
    index=0 
    ignored_chroms = []
    for line in list(data):
        line=line.strip('\n') 
        index+=1
        tokens=line.split('\t')
        chrom=tokens[0]
        gc=float(tokens[-1])
        start=tokens[1]
        end=tokens[2]
        chrom_real=chrom
        chrom = remap_chrom(chrom, splits_dict)
        if chrom == "ignore":
            ignored_chroms.append(chrom_real)
            continue
        if chrom not in gc_dict:
            gc_dict[chrom]={}
        if gc not in gc_dict[chrom]:
            gc_dict[chrom][gc]=[(chrom,start,end,chrom_real)]
        else:
            gc_dict[chrom][gc].append((chrom,start,end,chrom_real))

    return gc_dict

def scale_gc(cur_gc):
    """
    Randomly increase/decrease the gc-fraction value by 0.01
    """
    if random.random()>0.5:
        cur_gc+=0.01
    else:
        cur_gc-=0.01
    cur_gc=round(cur_gc,2)
    if cur_gc<=0:
        cur_gc+=0.01
    if cur_gc>=1:
        cur_gc-=0.01
    assert cur_gc >=0
    assert cur_gc <=1
    return cur_gc 

def adjust_gc(chrom,cur_gc,negatives,used_negatives):
    """
    Function that checks if (1) the given gc fraction value is available
    in the negative candidates or (2) if the given gc fraction value has 
    candidates not already sampled. If eitheir of the condition fails we  
    sample the neighbouring gc_fraction value by randomly scaling with 0.01.
    """
    if chrom  not in used_negatives:
        used_negatives[chrom]={}

    if cur_gc not in used_negatives[chrom]:
        used_negatives[chrom][cur_gc]=[]

    while (cur_gc not in negatives[chrom]) or (len(used_negatives[chrom][cur_gc])>=len(negatives[chrom][cur_gc])):
        cur_gc=scale_gc(cur_gc)
        if cur_gc not in used_negatives[chrom]:
            used_negatives[chrom][cur_gc]=[]
    return cur_gc,used_negatives 


def match_gc(args): 
    splits_dict=json.load(open(args.chr_fold_path))
    
    random.seed(args.seed)

    negatives=make_gc_dict(args.candidate_negatives, splits_dict)
    used_negatives=dict()
    cur_peaks=pd.read_csv(args.foreground_gc_bed,header=None,sep='\t')
    negatives_bed = []
    
    foreground_gc_vals = []
    output_gc_vals = []
    ignored_chroms = []
    for index,row in tqdm(list(cur_peaks.iterrows())): 

        chrom=row[0]
        start=row[1]
        end=row[2]
        gc_value=row[3]

        chrom_real=chrom
        chrom = remap_chrom(chrom, splits_dict)
        if chrom == "ignore":
            ignored_chroms.append(chrom_real)
            continue

        if chrom=="chrom_train" or chrom=="chrom_valid":
            neg_to_pos_ratio = args.neg_to_pos_ratio_train
        else:
            neg_to_pos_ratio = 1

        # for every gc value in positive how many negatives to find
        # we will keep the ratio of positives to negatives in the test set same
        for rep in range(neg_to_pos_ratio):
            cur_gc,used_negatives=adjust_gc(chrom,gc_value,negatives,used_negatives)
            num_candidates=len(negatives[chrom][cur_gc])
            rand_neg_index=random.randint(0,num_candidates-1)
            while rand_neg_index in used_negatives[chrom][cur_gc]:
                cur_gc,used_negatives=adjust_gc(chrom,cur_gc,negatives,used_negatives)
                num_candidates=len(negatives[chrom][cur_gc])
                rand_neg_index=random.randint(0,num_candidates-1)

            used_negatives[chrom][cur_gc].append(rand_neg_index)
            neg_tuple=negatives[chrom][cur_gc][rand_neg_index]
            neg_chrom=neg_tuple[0]
            neg_start=neg_tuple[1]
            neg_end=neg_tuple[2]
            neg_chrom_real=neg_tuple[3]
            negatives_bed.append([neg_chrom_real,int(neg_start),int(neg_end), cur_gc]) 
            output_gc_vals.append(cur_gc)
            foreground_gc_vals.append(gc_value)       
   
    negatives_bed = pd.DataFrame(negatives_bed)
    negatives_bed[3] = "."
    negatives_bed[4] = "."
    negatives_bed[5] = "."
    negatives_bed[6] = "."
    negatives_bed[7] = "."
    negatives_bed[8] = "."
    negatives_bed[9] = args.inputlen // 2
    negatives_bed.to_csv(args.output_prefix+".bed", sep='\t', index=False, header=False, quoting=csv.QUOTE_NONE)

    # checking how far the true distribution of foreground is compared to the backgrounds generated
    bins = np.linspace(0, 1, 100)
    plt.hist([output_gc_vals,foreground_gc_vals], bins, density=True, label=['negatives gc distribution', "foreground gc distribution"])
    plt.xlabel("GC content")
    plt.ylabel("Density")
    plt.legend(loc='upper right')
    plt.savefig(args.output_prefix+"_compared_with_foreground.png")

### Process peaks for bias model

In [ ]:
from chrombpnet.helpers.make_gc_matched_negatives.get_genomewide_gc_buckets.get_genomewide_gc_bins import get_genomewide_gc
import chrombpnet.helpers.make_gc_matched_negatives.get_gc_content as get_gc_content
import argparse
import os
import pandas as pd
import concurrent.futures
from tqdm.contrib.concurrent import process_map

In [ ]:
## Genome-wide GC content
# Define arguments
genome_fasta_path = f"{chrombpnet_dir}/data/hg38.fa"
output_directory = f"{chrombpnet_dir}/data/peaks/"

# Folder
if not os.path.exists(output_directory):
    os.makedirs(output_directory)
        
args = argparse.Namespace(
    genome = genome_fasta_path,
    stride = 1000,
    inputlen = 2114,
    output_prefix = output_directory + "genome_wide_gc.bed"
)
get_genomewide_gc(args.genome, args.output_prefix, args.inputlen, args.stride)

## GC content in peaks
chrom_size_file = f"{chrombpnet_dir}/data/hg38.chrom.sizes"
peak_file = f"{chrombpnet_dir}/data/peaks_no_blacklist_summits.bed"
blacklist_file = f"{chrombpnet_dir}/data/blacklist.bed"
args = argparse.Namespace(
    genome = genome_fasta_path,
    input_bed = peak_file,
    chrom_sizes = chrom_size_file,
    inputlen = 2114,
    output_prefix = output_directory + "all_peaks.gc"
)
get_gc_content.main(args)

# Resize peaks and import them
os.system("bedtools slop -i {peaks} -g {chrom_sizes} -b {flank_size} > {output}".format(peaks=args.input_bed, chrom_sizes=args.chrom_sizes, flank_size=args.inputlen//2, output=output_directory + "all_peaks_slop.bed"))
exclude_bed = pd.read_csv(output_directory + "all_peaks_slop.bed", sep="\t", header=None, usecols=[0,1,2])

# Resize blacklist, import concatenating to exclude list
os.system("bedtools slop -i {peaks} -g {chrom_sizes} -b {flank_size} > {output}".format(peaks=blacklist_file, chrom_sizes=args.chrom_sizes, flank_size=args.inputlen//2, output=output_directory + "blacklist_slop.bed"))
exclude_bed = pd.concat([exclude_bed,pd.read_csv(output_directory + "blacklist_slop.bed",sep="\t",header=None, usecols=[0,1,2])])

# Write to a file a sort
exclude_bed.to_csv(output_directory+"exclude_regions.unsorted.bed", sep="\t", header=False, index=False)
os.system("bedtools sort -i {inputb} | bedtools merge -i stdin > {output}".format(inputb=output_directory+"exclude_regions.unsorted.bed", output=output_directory+"exclude_regions.bed"))			

# Get candidate regions
os.system("bedtools intersect -v -a {genomewide_gc} -b {exclude_bed}  > {candidate_bed}".format(genomewide_gc=output_directory + "genome_wide_gc.bed",exclude_bed=output_directory+"exclude_regions.bed",candidate_bed=output_directory+"candidates.bed"))

In [ ]:
# Define function to run a fold
def process_fold(fold):
    input_directory = f"{chrombpnet_dir}/data/peaks/"
    
    # Set output directory for this fold
    output_directory = f"{chrombpnet_dir}/data/fold{fold}/peaks/bias/"
    
    # Ensure the output directory exists
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    # Setup arguments
    args = argparse.Namespace(
        candidate_negatives=input_directory + "candidates.bed",
        foreground_gc_bed=input_directory + "all_peaks.gc.bed",
        output_prefix=output_directory + "bias",
        chr_fold_path=f"{chrombpnet_dir}/data/fold_{fold}.json",
        neg_to_pos_ratio_train=1,
        seed=1234,
        inputlen = 2114
    )
    
    # Run the function (ensure get_gc_matched_negatives.main is available and callable)
    match_gc(args)

# Run folds in parallel
process_map(process_fold, range(5), max_workers=5)
 

### Get hyperparameters for bias models

In [ ]:
import chrombpnet.helpers.hyperparameters.find_bias_hyperparams as find_bias_hyperparams
import os
import argparse
import concurrent.futures
from tqdm.contrib.concurrent import process_map

In [ ]:
combined_bigWig = f"{chrombpnet_dir}/data/combined_unstranded.bw"
genome_fasta_path = f"{chrombpnet_dir}/data/hg38.fa"

# Define function to run in parallel
def process_fold(fold):  
    # Set output directory for this fold
    output_directory = f"{chrombpnet_dir}/data/fold{fold}/bias/"
    
    # Ensure the output directory exists
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    # Setup arguments
    args = argparse.Namespace(
        genome = genome_fasta_path,
        bigwig = combined_bigWig,
        peaks = f"{chrombpnet_dir}/data/peaks_no_blacklist_summits.bed",
        nonpeaks = f"{chrombpnet_dir}/data/fold{fold}/peaks/bias/bias.bed",
        bias_threshold_factor = 0.5,
        outlier_threshold = 0.9999,
        max_jitter = 50,
        chr_fold_path = f"{chrombpnet_dir}/data/fold_{fold}.json",
        inputlen = 2114,
        outputlen = 1000,
        filters = 128,
        n_dilation_layers = 4,
        output_prefix = output_directory,
    )
    
    # Run the function
    find_bias_hyperparams.main(args)

# Run folds in parallel
process_map(process_fold, range(5), max_workers=5) 

### Train the bias models

In [ ]:
import chrombpnet.training.models.bpnet_model as bpnet_model
import chrombpnet.training.train as train
import os
import argparse
import pandas as pd

In [ ]:
combined_bigWig = f"{chrombpnet_dir}/data/combined_unstranded.bw"
genome_fasta_path = f"{chrombpnet_dir}/data/hg38.fa"
for fold in range(5):
    # Make directory
    model_directory = f"{chrombpnet_dir}/data/fold" + str(fold) + "/bias/models/"
    if not os.path.exists(model_directory):
        os.makedirs(model_directory)

    # Set count loss weight to 5
    param_file = f"{chrombpnet_dir}/data/fold" + str(fold) + "/bias/bias_model_params.tsv"
    params = pd.read_csv(param_file, sep="\t", header=None)
    params.iloc[0,1] = str(5)
    params.to_csv(f"{chrombpnet_dir}/test_params.tsv", sep="\t", header=None, index=None)
        
    # Setup arguments
    args = argparse.Namespace(
        genome = genome_fasta_path,
        bigwig = combined_bigWig,
        peaks = "None",
        nonpeaks = f"{chrombpnet_dir}/data/fold" + str(fold) + "/bias/filtered.bias_nonpeaks.bed",
        output_prefix = model_directory + "bias",
        chr_fold_path = f"{chrombpnet_dir}/data/fold_" + str(fold) + ".json",
        epochs = 50,
        early_stop = 5,
        batch_size = 64,
        learning_rate = 0.001,
        params = f"{chrombpnet_dir}/test_params.tsv",
        trackables = ['loss','val_loss'],
        seed = 1234,
        architecture_from_file = bpnet_model.__file__
    )
    
    # Train the bias models
    train.main(args)

### Process peaks for accessibility models

In [ ]:
import argparse
import pysam
from multiprocessing import Pool, cpu_count
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from chrombpnet.helpers.make_gc_matched_negatives.get_genomewide_gc_buckets.get_genomewide_gc_bins import get_genomewide_gc
import chrombpnet.helpers.make_gc_matched_negatives.get_gc_content as get_gc_content
import argparse
import os
import pandas as pd
import concurrent.futures
from tqdm.contrib.concurrent import process_map
from tqdm.notebook import tqdm
import shutil
from functools import partial

def count_overlaps_chunk(args):
    peak_lines, fragment_file = args
    tabix_file = pysam.TabixFile(fragment_file)
    results = []

    for line in peak_lines:
        if line.startswith("#") or not line.strip():
            continue
        fields = line.strip().split('\t')
        chrom = fields[0]
        try:
            start = int(fields[1])
            end = int(fields[2])
        except (IndexError, ValueError):
            continue

        count = 0
        try:
            records = tabix_file.fetch(chrom, start, end)
            count = sum(1 for _ in records)
        except ValueError:
            count = 0

        results.append(line.strip() + '\t' + str(count) + '\n')
    return results

def chunkify(lst, n):
    """Split list lst into n approximately equal chunks"""
    k, m = divmod(len(lst), n)
    return (lst[i*k + min(i, m):(i+1)*k + min(i+1, m)] for i in range(n))

def count_overlaps(peak_file, fragment_file, output_file, threads):
    with open(peak_file) as f:
        lines = [line for line in f if line.strip() and not line.startswith("#")]

    # Make chunks
    chunks = list(chunkify(lines, threads))

    # Prepare args for each process
    args = [(chunk, fragment_file) for chunk in chunks]

    with Pool(processes=threads) as pool:
        results = pool.map(count_overlaps_chunk, args)

    # Flatten results and write
    with open(output_file, 'w') as fout:
        for chunk_result in results:
            fout.writelines(chunk_result)

In [ ]:
celltypes = ["acinar", "activated_stellate", "alpha", "beta", "cycling", "delta","ductal", "endothelial", "epsilon", "lymphoid", "mast", "myeloid", "pp", "schwann", "quiescent_stellate"]
genome_fasta_path = f"{chrombpnet_dir}/data/hg38.fa"
foreground_peaks = f"{chrombpnet_dir}/data/peaks_no_blacklist_summits.bed"
background_peaks = f"{chrombpnet_dir}/data/fold0/peaks/bias/bias.bed"
threads = 15

for celltype in tqdm(celltypes):
    # Define fragment file
    fragment_file = f"{data_dir}/scATACseq/ArchR/IsletProject/GroupFragments/fragments/" + str(celltype) + ".fragments.tsv.gz"

    # Define output files
    foreground_output = f"{chrombpnet_dir}/data/peaks/{celltype}_foreground.counts"
    background_output = f"{chrombpnet_dir}/data/peaks/{celltype}_background.counts"
    output_bed = f"{chrombpnet_dir}/data/peaks/{celltype}_filtered.bed"
    output_png = f"{chrombpnet_dir}/data/peaks/{celltype}_filtering.png"
    
    # Count overlaps
    count_overlaps(foreground_peaks, fragment_file, foreground_output, threads)
    count_overlaps(background_peaks, fragment_file, background_output, threads)

    # Read input files, no header, tab-separated
    background = pd.read_csv(background_output, sep="\t", header=None)
    foreground = pd.read_csv(foreground_output, sep="\t", header=None)

    # Calculate threshold: mean + multiplier * std() on column 10 (zero-based index 10)
    threshold = background[10].mean() + background[10].std() * 1.0

    # Filter foreground on threshold
    foreground_filtered = foreground[foreground[10] >= threshold]

    # Sort filtered foreground by chr (col0) then start (col1)
    foreground_filtered = foreground_filtered.sort_values(by=[0,1], ascending=[True, True])

    # Save filtered foreground to bed file
    foreground_filtered_to_save = foreground_filtered.drop(columns=10)
    foreground_filtered_to_save.to_csv(output_bed, sep="\t", header=False, index=False)

    # Prepare label for foreground with count of filtered rows
    peak_label = f"Peaks ({len(foreground_filtered)})"

    # Plot histogram
    plt.figure(figsize=(8,6))

    sns.histplot(foreground[10], color='green', label=peak_label, kde=False, stat="density", alpha=0.6)
    sns.histplot(background[10], color='red', label='Background', kde=False, stat="density", alpha=0.6)

    plt.axvline(threshold, color='blue', linestyle='--', label=f'Threshold = {threshold:.2f}')

    plt.legend()
    plt.xlabel('Counts')
    plt.ylabel('Density')
    plt.xscale('log')
    plt.tight_layout()
    plt.savefig(output_png)
    plt.close()


In [ ]:
# Define function to calculate GC content in peaks from a specific cell type
def process_celltype(celltype):
    output_path = f"{chrombpnet_dir}/data/peaks/{celltype}_gc"
    peak_file = f"{chrombpnet_dir}/data/peaks/{celltype}_filtered.bed"

    ## GC content in peaks
    chrom_size_file = f"{chrombpnet_dir}/data/hg38.chrom.sizes"
    blacklist_file = f"{chrombpnet_dir}/data/blacklist.bed"
    args = argparse.Namespace(
        genome = genome_fasta_path,
        input_bed = peak_file,
        chrom_sizes = chrom_size_file,
        inputlen = 2114,
        output_prefix = output_path
    )
    get_gc_content.main(args)

# Run celltypes in parallel
process_map(process_celltype, celltypes, max_workers=15) 

In [ ]:
# Create folders
for fold in range(5):
    output_directory = f"{chrombpnet_dir}/data/fold{fold}/peaks/accessibility/"
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
        
# Define function to run a fold
def process_celltype_fold(celltype, fold):  
    # Set outputs and inputs
    output_prefix = f"{chrombpnet_dir}/data/fold{fold}/peaks/accessibility/{celltype}_negatives"
    input_peaks = f"{chrombpnet_dir}/data/peaks/{celltype}_gc.bed"
    
    # Copy files to avoid blocking
    temp_candidates_file = f"/tmp/{celltype}_fold{fold}_candidates.bed"
    shutil.copy(f"{chrombpnet_dir}/data/peaks/candidates.bed", temp_candidates_file)

    # Setup arguments
    args = argparse.Namespace(
        candidate_negatives=temp_candidates_file,
        foreground_gc_bed=input_peaks,
        output_prefix=output_prefix,
        chr_fold_path=f"{chrombpnet_dir}/data/fold_{fold}.json",
        neg_to_pos_ratio_train=1,
        seed=1234,
        inputlen = 2114
    )
    
    # Run the function
    match_gc(args)
    os.remove(temp_candidates_file)

# Run celltypes and fold in parallel
for fold in range(5):
    process_map(partial(process_celltype_fold, fold=fold), celltypes, max_workers=15)

### Get hyperparameters for accessibility models

In [ ]:
import chrombpnet.helpers.hyperparameters.find_chrombpnet_hyperparams as find_chrombpnet_hyperparams
import os
import argparse
from tqdm.contrib.concurrent import process_map
from functools import partial

In [ ]:
# Source https://github.com/kundajelab/chrombpnet/blob/master/chrombpnet/helpers/hyperparameters/find_chrombpnet_hyperparams.py
# Seperate CPU and GPU tasks for better parallel processing

import argparse
import pyfaidx
import pyBigWig
import pandas as pd
import numpy as np
import os
from tensorflow import keras
import json
import numpy as np
import chrombpnet.training.utils.one_hot as one_hot
import tensorflow as tf
from tensorflow.keras.utils import get_custom_objects
from tensorflow.keras.models import load_model
import chrombpnet.training.utils.losses as losses

def filter_edge_regions(peaks_df, bw, width,):
    """
    Filter regions in bed file that are on edges i.e regions that cannot be used to construct
    input length + jitter length of the sequence
    """
    input_shape = peaks_df.shape[0]

    # left edge case
    filtered = np.array((peaks_df['start'] + peaks_df['summit'] - width//2) < 0)
    peaks_df = peaks_df[~filtered]
    num_filtered = sum(filtered)

    # right edge case
    chrom_to_sizes = bw.chroms()
    filtered = []
    for i, r in peaks_df.iterrows():
        if r['start'] + r['summit'] + width//2 > chrom_to_sizes.get(r['chr'], 0) :
            filtered.append(True)
        else:
            filtered.append(False)
    filtered=np.array(filtered)
    peaks_df = peaks_df[~filtered]
    num_filtered += sum(filtered)
    
    return peaks_df

def get_seqs_cts(genome, bw, peaks_df, input_width=2114, output_width=1000):
    """
    Output counts (not log counts)
    Output one-hot encoded sequence
    """
    vals = []
    seqs = []
    for i, r in peaks_df.iterrows():
        sequence = str(genome[r['chr']][(r['start']+r['summit'] - input_width//2):(r['start'] + r['summit'] + input_width//2)])
        seqs.append(sequence)
        bigwig_vals=np.nan_to_num(bw.values(r['chr'], 
                            (r['start'] + r['summit']) - output_width//2,
                            (r['start'] + r['summit']) + output_width//2))
        vals.append(bigwig_vals)
    return (np.sum(np.array(vals),axis=1), one_hot.dna_to_one_hot(seqs))

def load_model_wrapper(model_h5):
    # read .h5 model
    custom_objects={"tf": tf, "multinomial_nll":losses.multinomial_nll}    
    get_custom_objects().update(custom_objects)    
    model=load_model(model_h5)
    print("got the model")
    model.summary()
    return model
    

def parse_data_args():
    parser=argparse.ArgumentParser(description="find hyper-parameters for chrombpnet defined in src/training/models/chrombpnet_with_bias_model.py")
    parser.add_argument("-g", "--genome", type=str, required=True, help="Genome fasta")
    parser.add_argument("-i", "--bigwig", type=str, required=True, help="Bigwig of tn5 insertions. Ensure it is +4/-4 shifted")
    parser.add_argument("-p", "--peaks", type=str, required=True, help="10 column bed file of peaks. Sequences and labels will be extracted centered at start (2nd col) + summit (10th col).")
    parser.add_argument("-n", "--nonpeaks", type=str, required=True, help="10 column bed file of non-peak regions, centered at summit (10th column)")
    parser.add_argument("-sr", "--negative_sampling_ratio", type=float, default=0.1, help="Ratio of negatives to positive samples per epoch")
    parser.add_argument("-oth", "--outlier-threshold", type=float, default=0.9999, help="threshold to use to filter outlies")
    parser.add_argument("-j", "--max-jitter", type=int, required=True, default=500, help="Maximum jitter applied on either side of region (default 500 for chrombpnet model)")
    parser.add_argument("-fl", "--chr-fold-path", type=str, required=True, help="Fold information - dictionary with test,valid and train keys and values with corresponding chromosomes")
    return parser

def parse_model_args(parser):
    # arguments here defined the following model - src/training/models/chrombpnet_with_bias_model.py
    parser.add_argument("-il", "--inputlen", type=int, required=True, help="Sequence input length")
    parser.add_argument("-ol", "--outputlen", type=int, required=True, help="Prediction output length")
    parser.add_argument("-fil", "--filters", type=int, default=512, help="Number of filters to use in chrombpnet mode")
    parser.add_argument("-dil", "--n-dilation-layers", type=int, default=8, help="Number of dilation layers to use in chrombpnet model")
    parser.add_argument("-b", "--bias-model-path", type=str, required=True, help="path of bias model")
    parser.add_argument("-op", "--output-prefix", help="output prefix for storing hyper-param TSV for chrombpnet")
    args = parser.parse_args()
    return args

def adjust_bias_model_logcounts(bias_model, seqs, cts):
    """
    Given a bias model, sequences and associated counts, the function adds a 
    constant to the output of the bias_model's logcounts that minimises squared
    error between predicted logcounts and observed logcounts (infered from 
    cts). This simply reduces to adding the average difference between observed 
    and predicted to the "bias" (constant additive term) of the Dense layer.
    Typically the seqs and counts would correspond to training nonpeak regions.
    ASSUMES model_bias's last layer is a dense layer that outputs logcounts. 
    This would change if you change the model.
    """

    # safeguards to prevent misuse
    #assert(bias_model.layers[-1].name == "logcount_predictions")
    assert(bias_model.layers[-1].name == "logcounts" or bias_model.layers[-1].name == "logcount_predictions")
    assert(bias_model.layers[-1].output_shape==(None,1))
    assert(isinstance(bias_model.layers[-1], keras.layers.Dense))

    print("Predicting within adjust counts")
    _, pred_logcts = bias_model.predict(seqs, verbose=True)
    delta = np.mean(np.log(1+cts) - pred_logcts.ravel())

    dw, db = bias_model.layers[-1].get_weights()
    bias_model.layers[-1].set_weights([dw, db+delta])
    return bias_model


def get_hyper_params(args): 

    # read the fold information - we will evaluate hyperparams on the train+valid set and do nothing on the test set 
    splits_dict=json.load(open(args.chr_fold_path))
    chroms_to_keep=splits_dict["train"]+splits_dict["valid"]
    test_chroms_to_keep=splits_dict["test"]

    # read from bigwigw and fasta file
    bw = pyBigWig.open(args.bigwig) 
    genome = pyfaidx.Fasta(args.genome)

    # read peaks and non peaks    
    in_peaks =  pd.read_csv(args.peaks,
                           sep='\t',
                           header=None,
                           names=["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"])
    in_nonpeaks =  pd.read_csv(args.nonpeaks,
                           sep='\t',
                           header=None,
                           names=["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"])

    assert(in_peaks.shape[0] != 0) # peaks file is empty
    assert(in_nonpeaks.shape[0] !=0) # non peaks file is empty
    assert(args.inputlen >= args.outputlen) # inputlen should be greater than the outputlen 
                                            # inputlen and outlen are chosen based on the filters and dilations layers used

    # get train/valid peaks and test peaks seperately
    peaks = in_peaks[(in_peaks["chr"].isin(chroms_to_keep))]
    test_peaks = in_peaks[(in_peaks["chr"].isin(test_chroms_to_keep))]

    nonpeaks = in_nonpeaks[(in_nonpeaks["chr"].isin(chroms_to_keep))]
    test_nonpeaks = in_nonpeaks[(in_nonpeaks["chr"].isin(test_chroms_to_keep))]

    # step 1 filtering: filter peaks that are in the edges - which prevents us from making the inputlen regions - do this for all splits
    peaks = filter_edge_regions(peaks, bw, args.inputlen+2*args.max_jitter)
    test_peaks = filter_edge_regions(test_peaks, bw, args.inputlen)
   
    nonpeaks = filter_edge_regions(nonpeaks, bw, args.inputlen)
    test_nonpeaks = filter_edge_regions(test_nonpeaks, bw, args.inputlen)

    # step 2 filtering: filter peaks that are outliers in train and valid set - no filtering on test set
    peak_cnts, _ = get_seqs_cts(genome, bw, peaks, args.inputlen, args.outputlen)
    nonpeak_cnts, nonpeak_seqs = get_seqs_cts(genome, bw, nonpeaks, args.inputlen, args.outputlen)    
    assert(len(peak_cnts) == peaks.shape[0])
    assert(len(nonpeak_cnts) == nonpeaks.shape[0])


    if args.negative_sampling_ratio > 0:
        final_cnts = np.concatenate((peak_cnts,np.random.choice(nonpeak_cnts, replace=False, size=(int(args.negative_sampling_ratio*len(peak_cnts))))))
    else:
        final_cnts = peak_cnts

    upper_thresh = np.quantile(final_cnts, args.outlier_threshold)
    lower_thresh = np.quantile(final_cnts, 1-args.outlier_threshold)

    peaks = peaks[(peak_cnts< upper_thresh) & (peak_cnts>lower_thresh)]
    nonpeaks = nonpeaks[(nonpeak_cnts< upper_thresh) & (nonpeak_cnts>lower_thresh)]

    # combine train valid and test peak set and store them in a new file
    if nonpeaks.shape[0] > peaks.shape[0]:
        train_nonpeaks = nonpeaks.sample(n=peaks.shape[0], random_state=1)
    else:
        train_nonpeaks = nonpeaks
        
    frames = [peaks, test_peaks]
    all_peaks = pd.concat(frames)
    all_peaks.to_csv("{}filtered.peaks.bed".format(args.output_prefix), sep="\t",  header=False, index=False)
    frames = [train_nonpeaks, test_nonpeaks]
    all_nonpeaks = pd.concat(frames)
    all_nonpeaks.to_csv("{}filtered.nonpeaks.bed".format(args.output_prefix), sep="\t", header=False, index=False)

    # find counts loss weight for model training - using train and validation set
    counts_loss_weight = np.median(final_cnts[(final_cnts <= upper_thresh) & (final_cnts>=lower_thresh)])/10
    assert(counts_loss_weight != 0)
    #assert(counts_loss_weight > 1.0) # counts loss weight can go less than 1.0 if you have very low-read depth - make sure you have enough density in counts
                                     # check peak-calling

    if counts_loss_weight < 1.0:
        counts_loss_weight = 1.0
        print("WARNING: you are training on low-read depth data")

    # adjust bias model for training  - using train and validation set
    # the bias model might be trained on a difference read depth compared to the given data - so this step scales the bias model to account for that
    #bias_model = param_utils.load_model_wrapper(args.bias_model_path)
    #bias_model_scaled = adjust_bias_model_logcounts(bias_model, nonpeak_seqs[(nonpeak_cnts< upper_thresh) & (nonpeak_cnts>lower_thresh)], nonpeak_cnts[(nonpeak_cnts< upper_thresh) & (nonpeak_cnts>lower_thresh)])
    # save the new bias model
    #bias_model_scaled.save("{}bias_model_scaled.h5".format(args.output_prefix))

    # store the parameters being used  - in a TSV file
    file = open("{}chrombpnet_data_params.tsv".format(args.output_prefix),"w")
    file.write("\t".join(["counts_sum_min_thresh", str(round(lower_thresh,2))]))
    file.write("\n")
    file.write("\t".join(["counts_sum_max_thresh", str(round(upper_thresh,2))]))
    file.write("\n")
    file.write("\t".join(["trainings_pts_post_thresh", str(sum((final_cnts<upper_thresh) & (final_cnts>lower_thresh)))]))
    file.write("\n")
    file.close()

    file = open("{}chrombpnet_model_params.tsv".format(args.output_prefix),"w")
    file.write("\t".join(["counts_loss_weight", str(round(counts_loss_weight,2))]))
    file.write("\n")
    file.write("\t".join(["filters", str(args.filters)]))
    file.write("\n")
    file.write("\t".join(["n_dil_layers", str(args.n_dilation_layers)]))
    file.write("\n")
    file.write("\t".join(["bias_model_path", "{}bias_model_scaled.h5".format(args.output_prefix)]))
    file.write("\n")
    file.write("\t".join(["inputlen", str(args.inputlen)]))
    file.write("\n")
    file.write("\t".join(["outputlen", str(args.outputlen)]))
    file.write("\n")
    file.write("\t".join(["max_jitter", str(args.max_jitter)]))
    file.write("\n")
    file.write("\t".join(["chr_fold_path", str(args.chr_fold_path)]))
    file.write("\n")
    file.write("\t".join(["negative_sampling_ratio", str(args.negative_sampling_ratio)]))
    file.close()

    return (args.bias_model_path, nonpeak_seqs[(nonpeak_cnts< upper_thresh) & (nonpeak_cnts>lower_thresh)], nonpeak_cnts[(nonpeak_cnts< upper_thresh) & (nonpeak_cnts>lower_thresh)], args.output_prefix)

In [ ]:
celltypes = ["acinar", "activated_stellate", "alpha", "beta", "cycling", "delta","ductal", "endothelial", "epsilon", "lymphoid", "mast", "myeloid", "pp", "schwann", "quiescent_stellate"]

# Create folders
for fold in range(5):
    # Make output directory
    output_directory = f"{chrombpnet_dir}/data/fold" + str(fold) + "/accessibility/"
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
        
# Define function to run a fold
def process_celltype_fold(celltype, fold):     
    # Define arguments
    args = argparse.Namespace(
        genome = genome_fasta_path,
        bigwig = f"{chrombpnet_dir}/data/{celltype}_coverage_unstranded.bw",
        peaks = f"{chrombpnet_dir}/data/peaks/{celltype}_filtered.bed",
        nonpeaks = f"{chrombpnet_dir}/data/fold{fold}/peaks/accessibility/{celltype}_negatives.bed",
        negative_sampling_ratio = 0.1,
        outlier_threshold = 0.9999,
        max_jitter = 500,
        chr_fold_path = f"{chrombpnet_dir}/data/fold_{fold}.json",
        inputlen = 2114,
        outputlen = 1000,
        filters = 512,
        n_dilation_layers = 8,
        bias_model_path = f"{chrombpnet_dir}/data/fold{fold}/bias/models/bias.h5",
        output_prefix = f"{chrombpnet_dir}/data/fold{fold}/accessibility/{celltype}_"
    )

    # Run the function
    result = get_hyper_params(args)
    return result

# Run celltypes and fold in parallel
res = []
for fold in range(5):
    results = process_map(partial(process_celltype_fold, fold=fold), celltypes, max_workers=15)
    res.append(results)

In [ ]:
from tqdm.notebook import tqdm
import chrombpnet.helpers.hyperparameters.param_utils as param_utils
# Adjust bias models (not in parallel due to GPU usage)
for fold in range(5):
    results = res[fold]
    for i in tqdm(range(len(results))):
        bias_model = param_utils.load_model_wrapper(results[i][0])
        bias_model_scaled = adjust_bias_model_logcounts(bias_model, results[i][1], results[i][2])
        bias_model_scaled.save("{}bias_model_scaled.h5".format(results[i][3]))

### Train accessibility models

In [ ]:
import chrombpnet.training.models.chrombpnet_with_bias_model as chrombpnet_with_bias_model
import chrombpnet.training.train as train
import os
import argparse
import pandas as pd

In [ ]:
import tensorflow as tf

if tf.config.list_physical_devices('GPU'):
    print("GPU is available")
else:
    print("GPU not found")

In [ ]:
celltypes = ["alpha", "beta", "cycling", "delta","ductal", "endothelial", "epsilon", "lymphoid", "mast", "myeloid", "pp", "schwann", "quiescent_stellate", "acinar", "activated_stellate"]
genome_fasta_path = f"{chrombpnet_dir}/data/hg38.fa"
for celltype in celltypes:
    for fold in range(5):
        # Make output directory
        output_directory = f"{chrombpnet_dir}/data/fold" + str(fold) + "/accessibility/models/"
        if not os.path.exists(output_directory):
            os.makedirs(output_directory)

        # Check if the model has already been run
        model_check_path = output_directory + str(celltype) + "_nobias.h5"
        if not os.path.exists(model_check_path):
            # Set count loss weight to 5
            param_file = f"{chrombpnet_dir}/data/fold" + str(fold) + "/accessibility/" + str(celltype) + "_chrombpnet_model_params.tsv"
            params = pd.read_csv(param_file, sep="\t", header=None)
            params.iloc[0,1] = str(5)
            params.to_csv(f"{chrombpnet_dir}/test_params.tsv", sep="\t", header=None, index=None)
                
            # Define arguments
            args = argparse.Namespace(
                genome = genome_fasta_path,
                bigwig = f"{chrombpnet_dir}/data/" + str(celltype) + "_coverage_unstranded.bw",
                peaks = f"{chrombpnet_dir}/data/fold" + str(fold) + "/accessibility/" + str(celltype) + "_filtered.peaks.bed",
                nonpeaks = f"{chrombpnet_dir}/data/fold" + str(fold) + "/accessibility/" + str(celltype) + "_filtered.nonpeaks.bed",
                params = f"{chrombpnet_dir}/test_params.tsv",
                output_prefix = output_directory + str(celltype),
                chr_fold_path = f"{chrombpnet_dir}/data/fold_" + str(fold) + ".json",
                epochs = 50,
                early_stop = 5,
                batch_size = 64,
                learning_rate = 0.001,
                trackables =  ['loss','val_loss'],
                seed = 1234,
                architecture_from_file = chrombpnet_with_bias_model.__file__
            )        
    
            # Run function
            train.main(args)
        else:
            print("Model already trained. Cell type: "  + str(celltype) + ", Fold: " + str(fold)) 

### Predict

In [ ]:
import chrombpnet.training.train as load_model_wrapper
import argparse
import pandas as pd
import os
import scipy.stats
import numpy as np
import json
import h5py
import tensorflow as tf
import chrombpnet.training.utils.argmanager as argmanager
import chrombpnet.training.utils.losses as losses
import chrombpnet.training.metrics as metrics
import chrombpnet.training.data_generators.initializers as initializers
from tensorflow.keras.utils import get_custom_objects
from tensorflow.keras.models import load_model
import chrombpnet.training.data_generators.batchgen_generator as batchgen_generator
from chrombpnet.training.utils import data_utils
from chrombpnet.training.predict import predict_on_batch_wrapper, write_predictions_h5py
import seaborn as sns
from scipy.stats import spearmanr, pearsonr

In [ ]:
# Setup
counter = 0
celltypes = ["acinar", "activated_stellate", "alpha", "beta", "cycling", "delta","ductal", "endothelial", "epsilon", "lymphoid", "mast", "myeloid", "pp", "schwann", "quiescent_stellate"]
for celltype in celltypes:
    for fold in range(5):
        for mode in ['train', 'valid', 'test']:
            model_path = f"{chrombpnet_dir}/data/fold{fold}/accessibility/models/{celltype}.h5"
            if os.path.exists(model_path):
                chr_fold_path = f"{chrombpnet_dir}/data/fold_{fold}.json"
                peaks_path = f"{chrombpnet_dir}/data/peaks/{celltype}_filtered.bed"
                genome_fasta_path = f"{chrombpnet_dir}/data/hg38.fa"
                batch_size = 64
                bigwig_path = f"{chrombpnet_dir}/data/{celltype}_coverage_unstranded.bw"
                output_file = f"{chrombpnet_dir}/data/fold{fold}/accessibility/predictions/{celltype}.h5"
                
                # Define the model object
                custom_objects={"tf": tf, "multinomial_nll":losses.multinomial_nll}    
                get_custom_objects().update(custom_objects)    
                model=load_model(model_path, compile=False)
                
                # Create data generator
                splits_dict=json.load(open(chr_fold_path))
                
                # Peaks
                NARROWPEAK_SCHEMA = ["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"]
                peak_regions=pd.read_csv(peaks_path,header=None,sep='\t',names=NARROWPEAK_SCHEMA)
                chroms=splits_dict[mode]
                peak_regions=peak_regions[peak_regions["chr"].isin(chroms)]
                
                # Static params
                inputlen = 2114
                outputlen = 1000
                negative_sampling_ratio = 1.0
                max_jitter = 0
                add_revcomp = False
                shuffle_at_epoch_start = False
                nonpeak_regions = None
                
                # Create the generator
                test_generator=batchgen_generator.ChromBPNetBatchGenerator(
                                                    peak_regions=peak_regions,
                                                    nonpeak_regions=nonpeak_regions,
                                                    genome_fasta=genome_fasta_path,
                                                    batch_size=batch_size,
                                                    inputlen=inputlen,                                        
                                                    outputlen=outputlen,
                                                    max_jitter=max_jitter,
                                                    negative_sampling_ratio=negative_sampling_ratio,
                                                    cts_bw_file=bigwig_path,
                                                    add_revcomp=add_revcomp,
                                                    return_coords=True,
                                                    shuffle_at_epoch_start=shuffle_at_epoch_start
                                                    )
                
                # Predictions
                true_counts, profile_probs_predictions, true_counts_sum, counts_sum_predictions, coordinates = predict_on_batch_wrapper(model, test_generator)
                
                # Metrics
                mnll_pw, mnll_norm, jsd_pw, jsd_norm, jsd_rnd, jsd_rnd_norm, mnll_rnd, mnll_rnd_norm =  metrics.profile_metrics(true_counts,profile_probs_predictions)
                spearman_cor=spearmanr(true_counts_sum,counts_sum_predictions)[0]
                pearson_cor=pearsonr(true_counts_sum,counts_sum_predictions)[0]  
                mse=((true_counts_sum - counts_sum_predictions)**2).mean(axis=0)
        
                # Collect in a pandas dataframe
                res_tmp = pd.DataFrame({
                    'Celltype': celltype,
                    'Fold': fold,
                    'Type': mode,
                    'MSE': mse,
                    'Pearson': pearson_cor,
                    'Spearman': spearman_cor,
                    'JSD_pw': np.nanmedian(jsd_pw),
                    'JSD_norm': np.nanmedian(jsd_norm),
                    'JSD_rand_pw': np.nanmedian(jsd_rnd),
                    'JSD_rand_norm': np.nanmedian(jsd_rnd_norm)
                }, index = [counter])
        
                # Combine
                if counter == 0:
                    met = res_tmp
                else:
                    met = pd.concat((met, res_tmp))
    
                # Write to h5 file
                write_predictions_h5py(output_file, profile_probs_predictions, counts_sum_predictions, coordinates)
                met.to_csv(f"{chrombpnet_dir}/training_metrics.csv")
        
                # Counter
                counter += 1

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
%matplotlib inline
means = met.groupby(['Celltype','Type']).mean(numeric_only=True).reset_index()
plt.rcParams['font.family'] = 'sans-serif'
sns.barplot(y = 'Pearson', x = 'Celltype', hue='Type', data = met)
plt.show()